# SSC4: structural-only, dropout and no-dropout models

Compare three frozen T1-trained pancreas models at **100 epochs**: SSC4-only,
SSC4 + MRI with whole-raw-channel dropout, and SSC4 + MRI without dropout.
The two fusion models each have raw-enabled and raw-muted inference policies,
so the comparison has **three fitted models and five policies**.

SSC4 computes four radius-1, reflected 3×3 patch-similarity maps from the same
MRI. The fusion models append normalized raw MRI as feature 4. Muting zeros
only that fifth feature after SSC4 has been computed unchanged.

The comparison below reads small, checked-in scalar artifacts only: no weights,
MRI, downloads, training or inference. Later sections retain the SSC4-only
saved-prediction browser and optional one-scan inference.


## 1. Terminal100 validation comparison

Run the next cells from the repository environment. The reporting data live in
`notebooks/data/` and work without the optional local MRI demo bundle.
To open the notebook from the repository root:

```sh
uv run --locked --extra cpu --group ssc --group notebooks jupyter lab notebooks/ssc-domain-generalization.ipynb
```

All fits use **323 T1 TRAIN / 46 development VAL cases**, seed 1729 and
25,000 updates. No T2 images enter fitting. T1 scores average **46 cases**;
T2 scores average scans within **53 operational groups from 56 validation scans**,
then weight groups equally. These are full-volume Dice scores.

| Fitted model | Inference policy | T1 case Dice | T2 group Dice |
| --- | --- | ---: | ---: |
| SSC4-only | Four SSC channels | 0.6856 | 0.4977 |
| SSC4 + MRI, dropout p=0.5 | Raw enabled | 0.7451 | 0.0582 |
| Same dropout weights | Raw muted | 0.6747 | 0.4880 |
| SSC4 + MRI, no dropout | Raw enabled | 0.7645 | 0.0058 |
| Same no-dropout weights | Raw muted | 0.0164 | 0.0102 |

Each row uses the same policy on both modalities. A raw-enabled T1 score and
raw-muted T2 score cannot be combined into one policy. SSC4-only provides the
clearest source/target balance in these runs; training-seed reliability is untested.


In [ ]:
%matplotlib inline
import importlib.metadata as metadata
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from ml_biomed_project.ssc import explore

PROJECT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/ml_biomed_project").is_dir()
)
DEMO_DIR = PROJECT / "data" / "ssc-demo-validation"
CASE_ID = "CAD_T2_110"
RUN_INFERENCE = True
DEVICE = "auto"  # CUDA, then MPS, then CPU.

In [ ]:
COMPARISON_DIR = PROJECT / "notebooks" / "data"
comparison = json.loads((COMPARISON_DIR / "ssc4-terminal100-comparison.json").read_text())
POLICIES = [
    ("SSC4", "SSC4-only", "SSC4-only"),
    ("DROP_RAW1", "Dropout · raw enabled", "Dropout\nraw enabled"),
    ("DROP_RAW0", "Dropout · raw muted", "Dropout\nraw muted"),
    ("NODROP_RAW1", "No dropout · raw enabled", "No dropout\nraw enabled"),
    ("NODROP_RAW0", "No dropout · raw muted", "No dropout\nraw muted"),
]
summary = pd.DataFrame(
    [
        {
            "Policy": label,
            "T1 Dice · 46 cases": comparison["means"][key]["T1"]["Dice"],
            "T2 Dice · 53 groups": comparison["means"][key]["T2"]["Dice"],
            "T2 Dice · 56 scans (secondary)": comparison["means"][key]["T2"]["scan_Dice"],
        }
        for key, label, _ in POLICIES
    ]
).set_index("Policy")
display(summary.round(6))

policy_colors = ["#6b7280", "#168e99", "#168e99", "#d86600", "#d86600"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, domain, units in zip(axes, ["T1", "T2"], ["46 cases", "56 scans / 53 groups"]):
    for position, (key, _, _) in enumerate(POLICIES):
        value = comparison["means"][key][domain]["Dice"]
        ax.scatter(
            position,
            value,
            color=policy_colors[position],
            s=70,
            marker="o" if key.endswith("RAW0") else "s",
            zorder=3,
        )
        ax.annotate(
            f"{value:.4f}",
            (position, value),
            xytext=(0, 9),
            textcoords="offset points",
            ha="center",
            fontsize=9,
        )
    ax.set(
        title=f"{domain} development validation · {units}",
        ylim=(0, 1.05),
        xticks=range(5),
        xticklabels=[tick for _, _, tick in POLICIES],
        ylabel="Mean full-volume Dice",
    )
    ax.tick_params(axis="x", labelsize=9)
    ax.grid(axis="y", alpha=0.2)
fig.text(
    0.02,
    0.02,
    "One fit per recipe; exposed validation. T1 equal-case and T2 equal-group means. "
    "Paired uncertainty is reported separately below.",
    fontsize=8,
)
fig.tight_layout(rect=(0, 0.09, 1, 1))
plt.show()

### What changes when raw MRI is muted?

The dropout model loses some source accuracy but recovers useful T2 performance.
The no-dropout model collapses on T1 and remains near failure on T2: its T2
muting change is **+0.0044**, with a 95% interval **[−0.0046, +0.0151]**.
That is no demonstrated useful rescue. Poor muting does not prove SSC was unused;
feature mixing, normalization/activation and decision-margin changes remain possible.

The paired intervals below describe case/group uncertainty for these fixed fits:
10,000 bootstrap draws, seed 20261102, unadjusted 95% percentile intervals.
They do not measure variation across training seeds or establish equivalence.
No new bootstrap is calculated here.


In [ ]:
muting_rows = []
for recipe, label in [("DROP", "SSC4 + MRI, dropout p=0.5"), ("NODROP", "SSC4 + MRI, no dropout")]:
    for domain, units in [("T1", 46), ("T2", 53)]:
        if recipe == "DROP":
            result = comparison["cached_dropout_standard_contrasts"][f"{domain}_RAW0-RAW1"]
        else:
            result = comparison["paired_comparisons"][f"{domain}_NODROP_RAW0-NODROP_RAW1"]
        muting_rows.append(
            {
                "Fit": label,
                "Domain": domain,
                "Paired units": units,
                "Muted − enabled Dice": result["difference"],
                "Unadjusted 95% CI": f"[{result['ci95'][0]:+.4f}, {result['ci95'][1]:+.4f}]",
            }
        )
display(pd.DataFrame(muting_rows).set_index(["Fit", "Domain"]).round(4))

## 2. Source training over the same 100 epochs

These are native training/validation losses and **T1 validation patch Dice**,
not full-volume Dice or T2 results. Fusion-model patch validation retains raw MRI;
it does not evaluate their raw-muted policies.

Dashed lines mark pauses after dropout epoch 38 and no-dropout epochs 14 and 59.
Native asynchronous augmentation/global streams restarted; the trajectories are
not fully matched. SSC4 values come from its checkpoint logger; the two fusion
histories use the retained rounded log values. No new training or validation is run.


In [ ]:
history = pd.read_csv(COMPARISON_DIR / "ssc4-terminal100-training.csv")
recipe_labels = {
    "SSC4": "SSC4-only",
    "DROP": "SSC4 + MRI · dropout p=0.5",
    "NODROP": "SSC4 + MRI · no dropout",
}
recipe_colors = {"SSC4": "#6b7280", "DROP": "#168e99", "NODROP": "#d86600"}
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, column, ylabel in zip(
    axes,
    ["train_loss", "val_loss", "patch_Dice"],
    ["Native training loss", "Native T1 validation loss", "Native T1 patch Dice"],
):
    for recipe, label in recipe_labels.items():
        rows = history.loc[history.recipe == recipe].sort_values("completed_epoch")
        ax.plot(
            rows.completed_epoch,
            rows[column],
            color=recipe_colors[recipe],
            label=label,
            linewidth=1.4,
        )
    for recipe, epoch in [("NODROP", 14), ("DROP", 38), ("NODROP", 59)]:
        ax.axvline(
            epoch + 0.5, color=recipe_colors[recipe], linestyle="--", alpha=0.6, linewidth=0.9
        )
    ax.set(ylabel=ylabel, xlim=(1, 100))
    ax.grid(axis="y", alpha=0.2)
axes[0].set_title("Three source-only fits · matched 100-epoch horizon", loc="left")
axes[0].legend(frameon=False, fontsize=9)
axes[-1].set(
    xlabel="Completed epoch (1-based)", ylim=(0, 1), xticks=[1, 14, 25, 38, 50, 59, 75, 100]
)
for recipe, epoch in [("NODROP", 14), ("DROP", 38), ("NODROP", 59)]:
    name = "No-dropout" if recipe == "NODROP" else "Dropout"
    axes[-1].annotate(
        f"{name} pause {epoch}",
        (epoch + 0.5, 0.10),
        xytext=(3, 0),
        textcoords="offset points",
        rotation=90,
        va="bottom",
        fontsize=8,
        color=recipe_colors[recipe],
    )
fig.text(
    0.02,
    0.02,
    "Patch Dice is distinct from full-volume Dice. Raw is enabled during fusion-model "
    "patch validation; streams and pauses differ between fits.",
    fontsize=8,
)
fig.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

## 3. Released checkpoints and portable loaders

| Fitted model | Checkpoint | Model metadata / code | Release |
| --- | --- | --- | --- |
| SSC4-only · 4 inputs | [ssc4-final.pth](https://github.com/bapierro/ml-biomed-project/releases/download/ssc4-demo-v1/ssc4-final.pth) | [SSC4 demo bundle](https://github.com/bapierro/ml-biomed-project/releases/download/ssc4-demo-v1/ssc4-demo-v1.zip) | [ssc4-demo-v1](https://github.com/bapierro/ml-biomed-project/releases/tag/ssc4-demo-v1) |
| SSC4 + MRI, dropout p=0.5 · 5 inputs | [ssc4-raw-dropout-final.pth](https://github.com/bapierro/ml-biomed-project/releases/download/ssc4-raw-dropout-v1/ssc4-raw-dropout-final.pth) | [Portable bundle](https://github.com/bapierro/ml-biomed-project/releases/download/ssc4-raw-dropout-v1/ssc4-raw-dropout-v1.zip) | [ssc4-raw-dropout-v1](https://github.com/bapierro/ml-biomed-project/releases/tag/ssc4-raw-dropout-v1) |
| SSC4 + MRI, no dropout · 5 inputs | [ssc4-raw-no-dropout-final.pth](https://github.com/bapierro/ml-biomed-project/releases/download/ssc4-raw-no-dropout-v1/ssc4-raw-no-dropout-final.pth) | [Portable bundle](https://github.com/bapierro/ml-biomed-project/releases/download/ssc4-raw-no-dropout-v1/ssc4-raw-no-dropout-v1.zip) | [ssc4-raw-no-dropout-v1](https://github.com/bapierro/ml-biomed-project/releases/tag/ssc4-raw-no-dropout-v1) |

All are the audited terminal100 weights. Enabled/muted policies of a fusion model
use the **same checkpoint**. Each new release includes trusted hashes, exact
plans/dataset metadata, a portable five-input loader and the full validation
comparison. The original SSC4 demo ZIP contains historical test illustrations;
its weights are unchanged, and the validation comparison above is separate.

For either five-input checkpoint, use `code/ssc4_raw_inference.py` from its
portable bundle and explicitly choose `--raw-policy enabled` or `muted`.
The repository's existing `load_model` and browser below support **SSC4-only**;
they do not load the five-input fusion checkpoints. These reporting cells do
not download or load any checkpoint.


## 4. Optional SSC4-only saved-prediction browser

Sections 1–3 above need only the checked-in reporting files. The following
browser additionally needs a local `data/ssc-demo-validation` bundle. It still
shows SSC4-only predictions; the new fusion results above are scalar comparisons.
From the repository root, where the completed local study is available:

```sh
uv run --locked --extra cpu --group ssc --group notebooks python scripts/prepare_ssc_demo.py
uv run --locked --extra cpu --group ssc --group notebooks jupyter lab notebooks/ssc-domain-generalization.ipynb
```

The local export uses the unchanged SSC4 checkpoint, saved validation predictions and six illustrations. It contains all **102 scan scores** (46 T1 validation + 56 T2 validation). No new inference or downloads occur during export. A complete bundle works offline and can be shared as a folder. The earlier public demo release contains historical test results; this notebook uses the separate validation bundle.


In [ ]:
demo = explore.load_demo(DEMO_DIR)
display(demo.cases)

### SSC4-only browser model

The model used **323 T1 training cases**, **46 development-validation cases**,
100 epochs and seed 1729. The saved curves show stochastic validation patches;
the results below score complete native volumes. Native compound loss can be negative.


In [ ]:
m = demo.manifest["model"]
display(
    pd.Series(
        {
            "Model": m["method"],
            "Parameters": f"{m['parameters']:,}",
            "Training": f"{m['epochs']} epochs · {m['updates']:,} updates · seed {m['seed']}",
            "Input": f"{m['input_features']} self-similarity channels · patch {m['patch']}",
            "Checkpoint SHA256": m["checkpoint_sha256"],
        }
    ).to_frame("SSC4")
)

### SSC4-only saved validation scores

Source T1 development VAL: **0.6856 mean Dice across 46 scans**.
T2 development VAL: **0.4977 across 53 operational groups** (56 scans; scan mean **0.4899**).
Repeated scans are averaged within each filename-derived group first.
The publisher validation split has 73 scans; 15 T1-development identifier overlaps and two unresolved aliases are excluded. These scores use all 56 eligible validation scans, not just the illustrations. T1 training is unchanged; further test evaluation is reserved for the final comparison.


In [ ]:
display(explore.study_summary(demo.scores).round(4))
explore.plot_study(demo.scores)
plt.show()

### Browse reference and SSC4 prediction

Choose a case, plane and slice. Pink is the reference; teal is SSC4. Dashed pink
contours expose missed reference regions. Scores cover the whole volume.
These are fixed, selected illustrations, not a representative sample.

Controls read saved masks and never rerun inference. Display uses canonical array
coordinates and physical aspect; it does not deoblique images. Thick slices can
make orthogonal views coarse.


In [ ]:
explore.case_viewer(demo, initial_case=CASE_ID)

## 5. Optional SSC4-only one-scan inference

Set `RUN_INFERENCE=True`, then rerun the next cell. Native nnU-Net reads,
normalizes, resamples, tiles, mirrors and restores the original grid. The label is
used only afterwards for scoring. A new output and provenance sidecar are saved;
existing outputs are never overwritten. For another scan, pass its NIfTI path to
`predict_volume`; no reference is needed for prediction.


In [ ]:
if RUN_INFERENCE:
    from ml_biomed_project.ssc import load_model, predict_volume

    model = load_model(demo.model_dir, device=DEVICE)
    row = demo.row(CASE_ID)
    output = PROJECT / "results" / "ssc4-notebook" / f"{CASE_ID}.nii.gz"
    prediction = predict_volume(model, demo.path(row["image"]), output)
    replay = explore.load_case(
        demo.path(row["image"]),
        demo.path(row["reference"]),
        {"SSC4": (prediction, 1)},
        name=CASE_ID,
    )
    display(replay["scores"].round(4))
    explore.plot_case(replay)
    plt.show()
else:
    print("Saved predictions only. Set RUN_INFERENCE=True to predict one scan.")

## Provenance and limits

The comparison contains one fitted initialization per recipe, all trained on T1
only. Both reported cohorts are exposed development validation. Case/group
bootstrap intervals do not establish training-seed robustness, equivalence or a
unique causal mechanism. Raw-muted input was absent during no-dropout fitting.
Cached raw-enabled T1 and fresh raw-muted preprocessing are not asserted bitwise
identical; retained source Direction-only header exceptions remain documented.

Earlier test records remain historical; no test evaluation, fitting or new
inference was performed to build these comparison cells. Filename-derived
operational groups are not verified biological identities. The checked-in JSON
retains exact comparison/audit hashes, checkpoint identities and release links;
the CSV retains the 300 completed epoch records used above.

Data: [CrossPan](https://huggingface.co/datasets/linkai-peng/CrossPan).
Descriptor inspiration: [SSC (2013)](https://iacl.ece.jhu.edu/proceedings/miccai2013/papers/8149/81490187.pdf),
a different 3D registration descriptor. The optional browser/inference remains
SSC4-only. Set `RUN_INFERENCE = False` before running all cells if you want to
browse saved outputs without predicting a new scan.


In [ ]:
pd.Series(
    {
        name: metadata.version(name)
        for name in ["torch", "nnunetv2", "nibabel", "numpy", "matplotlib", "ipywidgets"]
    },
    name="Environment",
)